# Lab: Handcrafted Features on CIFAR-10 and Why CNNs Matter

This notebook is designed as a **guided lab solution**. Each section begins with the **task/questions**, followed by the code needed to answer them.

## Learning goals

By the end of this lab, students should be able to:

1. Load and inspect an image dataset such as **CIFAR-10**.
2. Extract multiple **handcrafted visual features**.
3. Train several classifiers on those features.
4. Compare results through tables, confusion matrices, and colored plots.
5. Reflect on the limits of handcrafted pipelines and explain why **CNNs** are often more effective.

## Feature sets used

- Raw pixels (baseline)
- HOG (Histogram of Oriented Gradients)
- LBP (Local Binary Patterns)
- Color histogram
- Combined handcrafted features

## Classifiers used

- SVM
- Random Forest
- Neural Network (MLPClassifier)

## Suggested interpretation

This is not only an accuracy exercise. The real point is to observe that the classical pipeline requires:

- manual feature design,
- careful preprocessing,
- repeated trial and error,
- and still struggles to capture spatial hierarchies in natural images.

That naturally motivates **deep learning models such as CNNs**, which learn features directly from data.

## Task 0 - Read the lab brief

Before running anything, make sure you understand the workflow.

### Questions / tasks

1. What is the difference between **handcrafted features** and **learned features**?
2. Why might CIFAR-10 be challenging for classical vision pipelines?
3. Why is a fair benchmark important when comparing classifiers?
4. What signs in the final results would suggest that CNNs are worth using?

In [ ]:
# Core imports
import os
import tarfile
import pickle
import urllib.request
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from skimage.color import rgb2gray
from skimage.feature import hog, local_binary_pattern
from skimage.transform import resize

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid')
np.random.seed(42)
print('Imports loaded successfully.')

## Task 1 - Download or load CIFAR-10

### Questions / tasks

1. Download the **CIFAR-10 Python version** if it is not already available locally.
2. Load the training and test batches into NumPy arrays.
3. Print the shape of the image arrays and label arrays.
4. Report the class names.
5. Explain what each image tensor dimension means.

In [ ]:
DATA_DIR = Path('cifar10_data')
CIFAR_TAR = DATA_DIR / 'cifar-10-python.tar.gz'
CIFAR_FOLDER = DATA_DIR / 'cifar-10-batches-py'
CIFAR_URL = 'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz'

DATA_DIR.mkdir(exist_ok=True)

def download_cifar10(url=CIFAR_URL, tar_path=CIFAR_TAR):
    if tar_path.exists():
        print(f'Archive already exists: {tar_path}')
        return
    print('Downloading CIFAR-10 archive...')
    urllib.request.urlretrieve(url, tar_path)
    print('Download complete.')

def extract_cifar10(tar_path=CIFAR_TAR, output_dir=DATA_DIR):
    if CIFAR_FOLDER.exists():
        print(f'Extracted folder already exists: {CIFAR_FOLDER}')
        return
    print('Extracting CIFAR-10 archive...')
    with tarfile.open(tar_path, 'r:gz') as tar:
        tar.extractall(path=output_dir)
    print('Extraction complete.')

def load_pickle(file_path):
    with open(file_path, 'rb') as f:
        return pickle.load(f, encoding='bytes')

def load_cifar10(folder=CIFAR_FOLDER):
    train_images, train_labels = [], []
    for i in range(1, 6):
        batch = load_pickle(folder / f'data_batch_{i}')
        X = batch[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
        y = np.array(batch[b'labels'])
        train_images.append(X)
        train_labels.append(y)

    test_batch = load_pickle(folder / 'test_batch')
    X_test = test_batch[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    y_test = np.array(test_batch[b'labels'])

    meta = load_pickle(folder / 'batches.meta')
    class_names = [name.decode('utf-8') for name in meta[b'label_names']]

    X_train = np.concatenate(train_images, axis=0)
    y_train = np.concatenate(train_labels, axis=0)
    return X_train, y_train, X_test, y_test, class_names

if not CIFAR_FOLDER.exists():
    if not CIFAR_TAR.exists():
        download_cifar10()
    extract_cifar10()

X_train_full, y_train_full, X_test_full, y_test_full, class_names = load_cifar10()

print('Training images shape:', X_train_full.shape)
print('Training labels shape:', y_train_full.shape)
print('Test images shape    :', X_test_full.shape)
print('Test labels shape    :', y_test_full.shape)
print('Class names          :', class_names)

## Task 2 - Explore the dataset visually

### Questions / tasks

1. Show at least one example image from each class.
2. Plot the class distribution of the training set.
3. Comment on image size and visual complexity.
4. Why are tiny natural images harder than simple grayscale digit datasets?

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle('One Example from Each CIFAR-10 Class', fontsize=18, fontweight='bold')

for class_id, ax in enumerate(axes.ravel()):
    idx = np.where(y_train_full == class_id)[0][0]
    ax.imshow(X_train_full[idx])
    ax.set_title(class_names[class_id], fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.show()

class_counts = pd.Series(y_train_full).value_counts().sort_index()
class_df = pd.DataFrame({
    'class_name': class_names,
    'count': class_counts.values
})

plt.figure(figsize=(12, 5))
plt.bar(class_df['class_name'], class_df['count'], color=plt.cm.tab10(np.arange(10)))
plt.title('Training Set Class Distribution', fontsize=16, fontweight='bold')
plt.xlabel('Class')
plt.ylabel('Number of images')
plt.xticks(rotation=30)
plt.show()

## Task 3 - Build a balanced working subset

The full dataset is large enough that some classical models may take a long time. For teaching purposes we create a **balanced subset**.

### Questions / tasks

1. Sample the same number of images per class from the training set and test set.
2. Create a manageable benchmark subset.
3. Verify that the subset is balanced.
4. Explain why balanced sampling helps when comparing models.

In [ ]:
TRAIN_PER_CLASS = 1000   # 10,000 training images total
TEST_PER_CLASS = 200     # 2,000 test images total
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

def balanced_subset(X, y, samples_per_class, random_state=42):
    rng_local = np.random.default_rng(random_state)
    selected_indices = []
    for class_id in np.unique(y):
        class_indices = np.where(y == class_id)[0]
        chosen = rng_local.choice(class_indices, size=samples_per_class, replace=False)
        selected_indices.extend(chosen.tolist())
    selected_indices = np.array(selected_indices)
    rng_local.shuffle(selected_indices)
    return X[selected_indices], y[selected_indices]

X_train, y_train = balanced_subset(X_train_full, y_train_full, TRAIN_PER_CLASS, RANDOM_STATE)
X_test, y_test = balanced_subset(X_test_full, y_test_full, TEST_PER_CLASS, RANDOM_STATE + 1)

print('Working training subset:', X_train.shape, y_train.shape)
print('Working test subset    :', X_test.shape, y_test.shape)

subset_counts = pd.Series(y_train).value_counts().sort_index()
display(pd.DataFrame({'class_name': class_names, 'train_count': subset_counts.values}))

## Task 4 - Create helper functions for handcrafted features

### Questions / tasks

1. Implement feature extraction for:
   - raw pixels,
   - HOG,
   - LBP histogram,
   - color histogram,
   - combined handcrafted features.
2. Keep feature extraction consistent for train and test data.
3. Print feature dimensionality for a sample image.
4. Why do different descriptors capture different visual properties?

In [ ]:
# HOG configuration
HOG_ORIENTATIONS = 9
HOG_PIXELS_PER_CELL = (4, 4)
HOG_CELLS_PER_BLOCK = (2, 2)
LBP_RADIUS = 1
LBP_POINTS = 8 * LBP_RADIUS
LBP_BINS = LBP_POINTS + 2
COLOR_HIST_BINS = 16


def extract_raw_pixels(image):
    return image.astype(np.float32).reshape(-1) / 255.0


def extract_hog_features(image):
    gray = rgb2gray(image)
    features = hog(
        gray,
        orientations=HOG_ORIENTATIONS,
        pixels_per_cell=HOG_PIXELS_PER_CELL,
        cells_per_block=HOG_CELLS_PER_BLOCK,
        block_norm='L2-Hys',
        feature_vector=True
    )
    return features.astype(np.float32)


def extract_lbp_features(image):
    gray = rgb2gray(image)
    lbp = local_binary_pattern(gray, P=LBP_POINTS, R=LBP_RADIUS, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, LBP_BINS + 1), density=True)
    return hist.astype(np.float32)


def extract_color_histogram(image, bins=COLOR_HIST_BINS):
    feats = []
    for ch in range(3):
        hist, _ = np.histogram(image[:, :, ch], bins=bins, range=(0, 255), density=True)
        feats.extend(hist)
    return np.array(feats, dtype=np.float32)


def extract_combined_features(image):
    return np.concatenate([
        extract_hog_features(image),
        extract_lbp_features(image),
        extract_color_histogram(image)
    ]).astype(np.float32)


FEATURE_FUNCTIONS = {
    'Raw Pixels': extract_raw_pixels,
    'HOG': extract_hog_features,
    'LBP': extract_lbp_features,
    'Color Histogram': extract_color_histogram,
    'Combined': extract_combined_features,
}

sample_image = X_train[0]
for name, func in FEATURE_FUNCTIONS.items():
    print(f'{name:16s} -> feature length = {len(func(sample_image))}')

## Task 5 - Visualize what handcrafted descriptors focus on

### Questions / tasks

1. Pick a few sample images.
2. Show the original image, grayscale image, HOG visualization, and LBP map.
3. Explain what each representation emphasizes.
4. Which useful information might be lost when we convert rich images into fixed descriptors?

In [ ]:
def hog_visualization(image):
    gray = rgb2gray(image)
    _, hog_image = hog(
        gray,
        orientations=HOG_ORIENTATIONS,
        pixels_per_cell=HOG_PIXELS_PER_CELL,
        cells_per_block=HOG_CELLS_PER_BLOCK,
        block_norm='L2-Hys',
        visualize=True,
        feature_vector=True
    )
    return gray, hog_image


def lbp_map(image):
    gray = rgb2gray(image)
    return local_binary_pattern(gray, P=LBP_POINTS, R=LBP_RADIUS, method='uniform')

example_indices = [0, 1, 2]
fig, axes = plt.subplots(len(example_indices), 4, figsize=(16, 10))
fig.suptitle('How Handcrafted Descriptors View the Same Image', fontsize=18, fontweight='bold')

for row, idx in enumerate(example_indices):
    image = X_train[idx]
    gray, hog_img = hog_visualization(image)
    lbp_img = lbp_map(image)

    axes[row, 0].imshow(image)
    axes[row, 0].set_title('Original')
    axes[row, 1].imshow(gray, cmap='gray')
    axes[row, 1].set_title('Grayscale')
    axes[row, 2].imshow(hog_img, cmap='inferno')
    axes[row, 2].set_title('HOG view')
    axes[row, 3].imshow(lbp_img, cmap='viridis')
    axes[row, 3].set_title('LBP map')

    for col in range(4):
        axes[row, col].axis('off')

plt.tight_layout()
plt.show()

## Task 6 - Extract feature matrices for the full benchmark subset

### Questions / tasks

1. Convert every image in the working train and test subsets into feature vectors.
2. Store the resulting feature matrices in a dictionary.
3. Report each feature matrix shape.
4. Which feature type has the highest dimension? Which has the lowest?

In [ ]:
def build_feature_matrix(images, extractor, name='feature'):
    features = [extractor(img) for img in images]
    matrix = np.vstack(features).astype(np.float32)
    print(f'{name:16s}: {matrix.shape}')
    return matrix

X_train_features = {}
X_test_features = {}

for feature_name, feature_fn in FEATURE_FUNCTIONS.items():
    X_train_features[feature_name] = build_feature_matrix(X_train, feature_fn, name=f'{feature_name} train')
    X_test_features[feature_name] = build_feature_matrix(X_test, feature_fn, name=f'{feature_name} test')

## Task 7 - Visualize feature spaces with PCA/t-SNE

### Questions / tasks

1. Select one or two feature sets.
2. Reduce them to 2D for visualization.
3. Plot the feature space using class colors.
4. Discuss whether classes appear clearly separated.
5. What does poor separation suggest about handcrafted features on CIFAR-10?

In [ ]:
tsne_feature_sets = ['HOG', 'Combined']
num_tsne_samples = 1500
plot_indices = rng.choice(len(X_train), size=num_tsne_samples, replace=False)

fig, axes = plt.subplots(1, len(tsne_feature_sets), figsize=(16, 6))
if len(tsne_feature_sets) == 1:
    axes = [axes]

for ax, feature_name in zip(axes, tsne_feature_sets):
    feats = X_train_features[feature_name][plot_indices]
    labels = y_train[plot_indices]

    scaled = StandardScaler().fit_transform(feats)
    reduced = PCA(n_components=min(50, scaled.shape[1]), random_state=42).fit_transform(scaled)
    embedding = TSNE(n_components=2, init='pca', learning_rate='auto', perplexity=30,
                     random_state=42).fit_transform(reduced)

    scatter = ax.scatter(
        embedding[:, 0], embedding[:, 1],
        c=labels, cmap='tab10', s=18, alpha=0.8
    )
    ax.set_title(f't-SNE of {feature_name} Features', fontsize=14, fontweight='bold')
    ax.set_xlabel('Component 1')
    ax.set_ylabel('Component 2')

handles, _ = scatter.legend_elements()
fig.legend(handles, class_names, loc='lower center', ncol=5, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()
plt.show()

## Task 8 - Define the classifiers for benchmarking

### Questions / tasks

1. Prepare the following classifiers:
   - SVM,
   - Random Forest,
   - Neural Network.
2. Use scaling where appropriate.
3. Keep the evaluation pipeline consistent across feature sets.
4. Why do some models need scaling more than others?

In [ ]:
MODELS = {
    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LinearSVC(C=1.0, max_iter=5000, dual='auto', random_state=42))
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=250,
        max_depth=None,
        n_jobs=-1,
        random_state=42
    ),
    'Neural Network': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(
            hidden_layer_sizes=(256, 128),
            activation='relu',
            solver='adam',
            learning_rate_init=0.001,
            max_iter=40,
            early_stopping=True,
            random_state=42
        ))
    ])
}

print('Models ready:', list(MODELS.keys()))

## Task 9 - Benchmark all feature/classifier combinations

### Questions / tasks

1. Train every classifier on every feature type.
2. Evaluate on the same held-out test subset.
3. Store accuracy, macro F1, and predictions.
4. Build a result table.
5. Which combination works best? Which works worst?

In [ ]:
from sklearn.metrics import f1_score

results = []
predictions = {}
reports = {}

for feature_name in FEATURE_FUNCTIONS.keys():
    Xtr = X_train_features[feature_name]
    Xte = X_test_features[feature_name]

    for model_name, model in MODELS.items():
        print(f'Training {model_name} on {feature_name} features...')
        model.fit(Xtr, y_train)
        y_pred = model.predict(Xte)

        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='macro')

        results.append({
            'Feature Set': feature_name,
            'Classifier': model_name,
            'Accuracy': acc,
            'Macro F1': f1
        })
        predictions[(feature_name, model_name)] = y_pred
        reports[(feature_name, model_name)] = classification_report(
            y_test, y_pred, target_names=class_names, output_dict=True
        )

results_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False).reset_index(drop=True)
results_df

## Task 10 - Compare benchmark results with colored plots

### Questions / tasks

1. Plot the benchmark scores using multiple colors.
2. Compare classifiers within each feature type.
3. Compare feature types within each classifier.
4. Identify patterns, not just the top score.
5. Which model seems most robust?

In [ ]:
plt.figure(figsize=(14, 6))
sns.barplot(data=results_df, x='Feature Set', y='Accuracy', hue='Classifier', palette='Set2')
plt.title('Accuracy by Feature Set and Classifier', fontsize=16, fontweight='bold')
plt.ylim(0, 1.0)
plt.xticks(rotation=20)
plt.legend(title='Classifier', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

pivot_acc = results_df.pivot(index='Feature Set', columns='Classifier', values='Accuracy')
plt.figure(figsize=(9, 5))
sns.heatmap(pivot_acc, annot=True, fmt='.3f', cmap='YlGnBu', linewidths=0.5)
plt.title('Accuracy Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## Task 11 - Inspect the best and worst confusion matrices

### Questions / tasks

1. Find the best-performing model.
2. Find a weaker-performing model.
3. Plot confusion matrices for both.
4. Which classes are most often confused?
5. Why might handcrafted descriptors struggle with those pairs?

In [ ]:
best_row = results_df.iloc[0]
worst_row = results_df.iloc[-1]

best_key = (best_row['Feature Set'], best_row['Classifier'])
worst_key = (worst_row['Feature Set'], worst_row['Classifier'])

pairs_to_plot = [
    ('Best Model', best_key),
    ('Weaker Model', worst_key),
]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, (label, key) in zip(axes, pairs_to_plot):
    cm = confusion_matrix(y_test, predictions[key])
    sns.heatmap(cm, annot=False, cmap='magma', ax=ax, cbar=True)
    ax.set_title(f"{label}: {key[0]} + {key[1]}", fontsize=14, fontweight='bold')
    ax.set_xlabel('Predicted class')
    ax.set_ylabel('True class')
    ax.set_xticks(np.arange(len(class_names)) + 0.5)
    ax.set_yticks(np.arange(len(class_names)) + 0.5)
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.set_yticklabels(class_names, rotation=0)

plt.tight_layout()
plt.show()

print('Best model:')
display(best_row.to_frame().T)
print('\nWeaker model:')
display(worst_row.to_frame().T)

## Task 12 - Rank the results and write short interpretations

### Questions / tasks

1. Rank all combinations from best to worst.
2. Identify whether feature choice or classifier choice had a larger effect.
3. Write 3-5 observations from the benchmark.
4. Do the best classical results look strong enough for real-world use on natural images?

In [ ]:
display(results_df)

feature_summary = results_df.groupby('Feature Set')[['Accuracy', 'Macro F1']].mean().sort_values('Accuracy', ascending=False)
classifier_summary = results_df.groupby('Classifier')[['Accuracy', 'Macro F1']].mean().sort_values('Accuracy', ascending=False)

print('Average performance by feature set')
display(feature_summary)

print('Average performance by classifier')
display(classifier_summary)

## Task 13 - Compare handcrafted pipelines with the CNN idea

### Questions / tasks

Use the results above and answer the following:

1. What are the main weaknesses of handcrafted features on CIFAR-10?
2. What extra effort is needed in the classical pipeline?
3. Why can CNNs often outperform these methods on natural images?
4. Which properties of CNNs directly address the limitations observed here?

In [ ]:
comparison_points = pd.DataFrame({
    'Handcrafted pipeline': [
        'Manual feature engineering is required',
        'Color, texture, and shape cues must be selected by the designer',
        'Spatial hierarchy is only partially captured',
        'Multiple separate preprocessing steps are needed',
        'Performance often plateaus on complex image data'
    ],
    'CNN-based pipeline': [
        'Features are learned directly from data',
        'Early layers can detect edges and textures automatically',
        'Deeper layers capture parts, shapes, and object structure',
        'End-to-end optimization reduces manual design effort',
        'Usually scales better to natural image recognition tasks'
    ]
})
comparison_points

## Task 14 - Optional extension: estimate a classical ceiling and discuss next steps

### Questions / tasks

1. Increase the subset size and observe whether performance improves enough.
2. Tune one classifier hyperparameter.
3. Add one more handcrafted descriptor.
4. Reflect on whether the added complexity is worth it.
5. Write a short conclusion that motivates CNNs.

This section is intentionally open-ended so students can continue the experiment.

In [ ]:
# Optional experiment template
# Uncomment and modify if you want to scale up the benchmark.

# TRAIN_PER_CLASS = 2000
# TEST_PER_CLASS = 400
# Change one model hyperparameter and rerun the pipeline.
# Example:
# MODELS['Random Forest'] = RandomForestClassifier(
#     n_estimators=500,
#     max_depth=None,
#     n_jobs=-1,
#     random_state=42
# )

print('Optional extension cell ready.')

## Final reflection

A typical pattern on CIFAR-10 is that **classical handcrafted pipelines can work reasonably well**, but they often:

- require significant manual effort,
- remain sensitive to descriptor design,
- struggle with small, colorful, natural images,
- and fail to learn increasingly abstract visual concepts automatically.

That is exactly where **CNNs** become attractive. Instead of forcing the designer to invent the right descriptor, CNNs learn useful filters and representations from the data itself. In practice, that usually leads to better accuracy and a more scalable workflow on image recognition problems.

---

## Suggested discussion questions for students

1. Which handcrafted feature seemed most informative here, and why?
2. Which classifier benefited most from stronger features?
3. Which classes were hardest to separate, and what visual similarities explain that?
4. If you were building a real image recognition system, would you stop at this pipeline or move to CNNs?
5. What trade-off exists between interpretability, engineering effort, and performance?